<h1 style="color:red;">I. Exploration des données</h1>

In [1]:
#importation des bibliothèques nécessaires
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import math

### 1) Chargement des données

In [4]:
# Chargement du fichier Excel 
file_path = "BDD_IMG_IVG.xlsx"
df_img = pd.read_excel(file_path, sheet_name="IMG", header=1)
df_ivg = pd.read_excel(file_path, sheet_name="IVG", header=1) 

TypeError: read_excel() missing 1 required positional argument: 'io'

### 2) Description des données

<h4 style="color:red;">Fichier IMG :</h4>

Ce fichier regroupe les données des patientes ayant sollicité une Interruption Médicale de Grossesse (IMG), fondée sur des motifs maternels non somatiques. Il s’agit principalement de situations dans lesquelles la demande repose sur des facteurs psychologiques, sociaux ou physiques, sans présence de pathologie physique menaçant directement la santé de la mère.

<h4 style="color:red;">Fichier IVG :</h4>

Ce fichier rassemble les informations des patientes ayant eu recours à une Interruption Volontaire de Grossesse (IVG). Cela inclut des contextes variés, qu’ils soient personnels, sociaux, économiques ou médicaux.

## Données IVG

### 3) Aperçu général des données

In [ ]:
# Afficher le nombre de lignes et de colonnes
print("Nombre de lignes :", df_ivg.shape[0])  
print("Nombre de colonnes :", df_ivg.shape[1])  

# Afficher le nom des colonnes
print("\nVariables présentes dans le dataset :")
print(df_ivg.columns.tolist())  

# Afficher les types de données de chaque colonne
print("\nTypes de données des colonnes :")
print(df_ivg.dtypes)

# Vérifier s'il y a des valeurs manquantes dans les colonnes
print("\nValeurs manquantes par colonne :")
print(df_ivg.isnull().sum())

# Afficher le pourcentage de valeurs manquantes
print("\nPourcentage de valeurs manquantes par colonne :")
print((df_ivg.isnull().mean() * 100).round(2).sort_values(ascending=False))

# Afficher les 10 premières lignes du dataset
df_ivg.head(10)

<h5 style="color:red;">Remarque : Les résultats montrent qu’il n’y a pas de valeurs manquantes. En revanche, certaines colonnes sont de type object (texte), alors qu’elles devraient contenir uniquement des entiers. Cela est probablement dû à la présence de valeurs mixtes (texte et chiffres) au sein d’une même colonne.</h5>

### 4) Analyse des valeurs manquantes

*Pour cela, nous allons examiner les modalités présentes dans chaque colonne contenant à la fois du texte et des chiffres, afin d’identifier les éventuelles incohérences de format.*

In [ ]:
# Nombre total de lignes (pour le calcul du %)
total = len(df_ivg)

# Parcourir les colonnes texte
for col in df_ivg.select_dtypes(include='object').columns:
    print(f"\n Colonne : {col}")
    print(" Valeurs uniques :", df_ivg[col].unique())

    # Nombre de "NR"
    nb_nr = (df_ivg[col] == "NR").sum()
    pct_nr = (nb_nr / total) * 100

    print(f" 'NR' : {nb_nr} valeurs ({pct_nr:.2f}%)")

<h5 style="color:red;"> On peut remarquer la présence de données manquantes notées "NR" dans certaines colonnes numériques. De plus, plusieurs modalités apparaissent parfois regroupées dans une même cellule, comme par exemple : "1+2+3", ce qui nécessite un traitement particulier pour les futures analyses. </h5>

### 5) Nettoyage des données

In [ ]:
# Colonnes à exploser
colonnes_multi = [
    "description_type_atcd_violences_ivg",
    "contexte_atcd_violence_ivg",
    "type_violence_actuelle_ivg"
]

# Nettoyage + split + explode successif
for col in colonnes_multi:
    df_ivg[col] = df_ivg[col].astype(str).str.replace(" ", "").str.split("+")
    df_ivg = df_ivg.explode(col)

# Affichage d’un extrait
print(df_ivg[["ivg_no_inclus",
                    "description_type_atcd_violences_ivg",
                    "contexte_atcd_violence_ivg",
                    "type_violence_actuelle_ivg"]].head(20))

In [ ]:
# Créer une fonction qui garde une seule valeur par ivg_no_inclus et laisse vide les autres
def remove_duplicates(df_ivg):
    df_unique = df_ivg.groupby('ivg_no_inclus', as_index=False).first()
    
    # Remplir les valeurs vides (NaN) avec un espace vide 
    df_unique = df_unique.fillna('')
    
    return df_unique

# Appliquer la fonction au DataFrame
df_cleaned = remove_duplicates(df_ivg)

# Afficher le résultat
print(df_cleaned[["ivg_no_inclus", 
                  "description_type_atcd_violences_ivg", 
                  "contexte_atcd_violence_ivg", 
                  "type_violence_actuelle_ivg"]].head(20))

In [ ]:
# Remplacer 'NR' par NaN dans les colonnes concernées
df_ivg.replace("NR", np.nan, inplace=True)

# Conversion en date
df_ivg["date_decouverte_grossesse_ivg"] = pd.to_datetime(df_ivg["date_decouverte_grossesse_ivg"], errors='coerce')

# Conversion en numériques entiers 
colonnes_entiers = [
    "description_type_atcd_violences_ivg",
    "type_violence_actuelle_ivg",
    "contexte_atcd_violence_ivg",
    "profession_tot_ivg",
    "pays_naiss_ivg",
    "situation_administrative_ivg",
    "violences_actuelles_ivg"
]
df_ivg[colonnes_entiers] = df_ivg[colonnes_entiers].apply(pd.to_numeric, errors='coerce').astype("Int64")

<h5 style="color:red;"> Pour le nettoyage des données, j’ai remplacé les valeurs "NR" par NaN, et pour les colonnes contenant plusieurs modalités dans une même cellule (par exemple "1+2+3"), j’ai séparé ces modalités et dupliqué les lignes correspondantes à l’aide de la méthode explode. </h5>

In [ ]:
# Afficher les types de données de chaque colonne
print("\nTypes de données des colonnes :")
print(df_ivg.dtypes)

In [ ]:
# Fichier IVG néttoyé
df_ivg.to_excel("IVG_nettoye.xlsx", index=False)

In [ ]:
# Chargement du fichier nettoyé
file_path = "IVG_nettoye.xlsx"
df_ivg = pd.read_excel(file_path, sheet_name="IVG")

### 6) Analyse des valeurs aberrantes ( distribution des variables et statistiques)

In [ ]:
# Colonnes numériques
cols = df_ivg.select_dtypes(include='number').columns

# Supprimer les doublons sur ivg_no_inclus
df_unique = df_ivg.drop_duplicates(subset='ivg_no_inclus')

# Nombre de lignes à afficher (2 histogrammes par ligne)
n = len(cols)
rows = math.ceil(n / 2)

# Affichage des histogrammes
df_unique[cols].hist(layout=(rows, 2), figsize=(10, rows * 4), xrot=25)
plt.tight_layout()
plt.show()

In [ ]:
# On enlève les doublons sur 'ivg_no_inclus' et garde le premier
df_ivg_unique = df_ivg.drop_duplicates(subset='ivg_no_inclus')

### 7) Analyse des corrélations

In [ ]:
# Statistiques récapitulatives des caractéristiques numériques
df_ivg_unique.describe()

In [ ]:
#Calcul de la matrice de corrélation
corrs = df_ivg_unique.select_dtypes(include='number').corr()
corrs

In [ ]:
#Affichage de la Heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(corrs, annot=True, cmap='coolwarm')
plt.title("Matrice de corrélation - IMG")
plt.show()

In [ ]:
# Filtrage des corrélations supérieures à 0.5 (hors diagonale)
high_corr = corrs.where(np.triu(np.ones(corrs.shape), k=1).astype(bool))
high_corr_filtered = high_corr[(high_corr.abs() > 0.5)]

# Récupération des couples de variables concernées
result = high_corr_filtered.stack().reset_index()
result.columns = ['Variable 1', 'Variable 2', 'Corrélation']

# Affichage
print(result)

## Données IMG

### 3) Aperçu général des données

In [ ]:
# Afficher le nombre de lignes et de colonnes
print("Nombre de lignes :", df_img.shape[0])  
print("Nombre de colonnes :", df_img.shape[1])  

# Afficher le nom des colonnes
print("\nVariables présentes dans le dataset :")
print(df_ivg.columns.tolist())  

# Afficher les types de données de chaque colonne
print("\nTypes de données des colonnes :")
print(df_img.dtypes)

# Vérifier s'il y a des valeurs manquantes dans les colonnes
print("\nValeurs manquantes par colonne :")
print(df_img.isnull().sum())

# Afficher le pourcentage de valeurs manquantes
print("\nPourcentage de valeurs manquantes par colonne :")
print((df_img.isnull().mean() * 100).round(2).sort_values(ascending=False))

# Afficher les 10 premières lignes du dataset
df_img.head(10)

<h5 style="color:red;">On remarque que certaines colonnes contiennent des valeurs manquantes (vides) et que, comme dans le fichier IVG, plusieurs colonnes sont de type object (texte), alors qu’elles devraient être de type numérique. Cela est probablement dû à la présence de chaînes de caractères comme "NR" ou à un mélange de texte et de chiffres dans les mêmes colonnes.</h5>

### 4) Analyse des valeurs manquantes

*Pour cela, nous allons examiner les modalités présentes dans chaque colonne contenant à la fois du texte et des chiffres, afin d’identifier les éventuelles incohérences de format.*

In [ ]:
# Liste des colonnes à analyser
colonnes = [
    "profession_simpl_img", "parcours_migratoire_img", "pays_naiss_img",
    "situation_administrative_img", "atcd_violence_img", "description_type_atcd_violences_img",
    "contexte_atcd_violence_img", "violences_actuelles_img", "type_violence_actuelle_img",
    "contexte_violence_actuelle_img", "grossesse_issue_viol_img", "nbr_atcd_ivg_img",
    "contraception_en_cours_img", "echec_contraception_urg_img", "grossesse_desire_img",
    "cycles_regluliers_img", "date_decouverte_grossesse_img", "terme_decouverte_grossesse_img",
    "nbr_jour_amenorrhée_découverte_img", "date_1ere_echo_img", "terme_1ere_echo_img",
    "nja_1ere_echo_img", "date_1ere_demande_img", "terme_1ere_demande_img", "nja_1ere_demande_img",
    "decision_pluridisciplinaire_img", "date_img", "terme_img", "nja_img", "lieu_img"
]

# Afficher modalités et pourcentages
for col in colonnes:
    print(f"\n Colonne : {col}")
    valeurs = df_img[col].astype(str).str.strip().fillna("NaN")
    freqs = valeurs.value_counts(dropna=False)
    pourcents = valeurs.value_counts(normalize=True, dropna=False) * 100

    for val in freqs.index:
        print(f"  '{val}' : {freqs[val]} ({pourcents[val]:.2f}%)")

<h5 style="color:red;"> On peut remarquer la présence de valeurs manquantes, représentées sous différentes formes comme des cellules vides, des "NR" ou des "NA". Certaines colonnes contiennent également un mélange de texte et de chiffres, voire uniquement du texte, alors qu’elles devraient contenir des valeurs numériques. </h5>

### 5) Nettoyage des données

In [ ]:
# Remplacement dans la colonne 'description_type_atcd_violences_img'
df_img['description_type_atcd_violences_img'] = df_img['description_type_atcd_violences_img'].replace({
    '1+2+4 + prostitution': '1+2+4+7',
    'ASE ds l\'enfance + fugues': '6'
})

# Remplacement dans la colonne 'contexte_atcd_violence_img'
df_img['contexte_atcd_violence_img'] = df_img['contexte_atcd_violence_img'].replace({
    '7 (dans un gang, drogue + prostitution)': '7',
    'ASE ds l\'enfance + fugues': '8'
})

In [ ]:
# Nettoyage de la colonne nbr_atcd_ivg_img
df_img['nbr_atcd_ivg_img'] = df_img['nbr_atcd_ivg_img'].replace({
    '1 IMG': '1',
    '1 IMG foetale': '1',
    'nan': np.nan
})

In [ ]:
# Nettoyage de la colonne echec_contraception_urg_img
df_img['echec_contraception_urg_img'] = df_img['echec_contraception_urg_img'].replace({
    'echec IVG': '2',
    'echec IVG med': '2',
    'nan': np.nan
})

In [ ]:
# La colonne IMG ( oui  à été remplacé par 1 pour ceux ayant réalisé et 0 le reste (non,hollande,changement,perdu)

In [ ]:
# Remplacer 'NR' par NaN dans les colonnes concernées
df_img.replace("NR", np.nan, inplace=True)

In [ ]:
# Conversion en numériques entiers
colonnes_a_convertir = [
    'profession_simpl_img',
    'parcours_migratoire_img',
    'pays_naiss_img',
    'situation_administrative_img',
    'grossesse_issue_viol_img',
    'nbr_atcd_ivg_img',
    'contraception_en_cours_img',
    'echec_contraception_urg_img',
    'grossesse_desire_img',
    'cycles_regluliers_img',
    'nbr_jour_amenorrhée_découverte_img',
    'nja_1ere_echo_img',
    'nja_1ere_demande_img',
    'decision_pluridisciplinaire_img',
    'nja_img',
    'lieu_img',
    'atcd_violence_img',                          
    'description_type_atcd_violences_img',            
    'contexte_atcd_violence_img',                   
    'violences_actuelles_img',                        
    'type_violence_actuelle_img',                     
    'contexte_violence_actuelle_img'
]

for col in colonnes_a_convertir:
    df_img[col] = pd.to_numeric(df_img[col], errors='coerce').astype('Int64')

In [ ]:
# vérification des types de données après nettoyage
df_img.dtypes

<h5 style="color:red;">J’ai procédé au nettoyage des données en remplaçant les valeurs "NA" et "NR" par NaN. J’ai ensuite utilisé la méthode explode() pour dupliquer les lignes lorsque plusieurs modalités étaient présentes dans une même cellule, et j’ai séparé les modalités exprimées sous forme de texte afin de pouvoir les analyser individuellement. </h5>

In [ ]:
# Fichier IMG néttoyé
df_img.to_excel("IMG_nettoye.xlsx", index=False)

In [ ]:
# Chargement du fichier nettoyé
file_path = "IMG_nettoye.xlsx"
df_img = pd.read_excel(file_path, sheet_name="IMG")

### 6) Analyse des valeurs aberrantes ( distribution des variables et statistiques)

In [ ]:
# Colonnes numériques
cols = df_img.select_dtypes(include='number').columns

# Supprimer les doublons sur ivg_no_inclus
df_unique = df_img.drop_duplicates(subset='no_inclus')

# Nombre de lignes à afficher (2 histogrammes par ligne)
n = len(cols)
rows = math.ceil(n / 2)

# Affichage des histogrammes
df_unique[cols].hist(layout=(rows, 2), figsize=(10, rows * 4), xrot=25)
plt.tight_layout()
plt.show()

In [ ]:
# On enlève les doublons sur 'ivg_no_inclus' et garde le premier
df_img_unique = df_img.drop_duplicates(subset='no_inclus')

In [ ]:
# Statistiques récapitulatives des caractéristiques numériques
df_img_unique.describe()

In [ ]:
# Statistiques récapitulatives des caractéristiques catégoriques
df_img_unique.describe(include='object')

### 7) Analyse des corrélations

In [ ]:
# Calcul de la matrice de corrélation
corrs = df_img_unique.select_dtypes(include='number').corr()
corrs

In [ ]:
#Affichage de la heatmap
plt.figure(figsize=(16, 8))
sns.heatmap(corrs, annot=True, cmap='coolwarm')
plt.title("Matrice de corrélation - IMG")
plt.show()

In [ ]:
# Filtrage des corrélations supérieures à 0.5 (hors diagonale)
high_corr = corrs.where(np.triu(np.ones(corrs.shape), k=1).astype(bool))
high_corr_filtered = high_corr[(high_corr.abs() > 0.45)]

# Récupération des couples de variables concernées
result = high_corr_filtered.stack().reset_index()
result.columns = ['Variable 1', 'Variable 2', 'Corrélation']

# Affichage
print(result)